# Football Player Physical Data — Mini Case Study

**Course:** Physical Data Visualization | Dr. Mostafa Mostafa | 4th Year CS

This notebook addresses all **8 analysis questions** from the case study using GPS wearable tracking data from a football training session. Each section includes the code, the visualization, the encoding rationale, and interpretation.

## 1 · Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('Done')


## 2 · Data Loading & Preparation

In [ ]:
df = pd.read_csv('dataset.csv')
print(f'Shape: {df.shape}')
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df.sort_values('Timestamp').reset_index(drop=True)
df['Elapsed_min'] = (df['Timestamp'] - df['Timestamp'].min()).dt.total_seconds() / 60.0
df.head(3)


### 2.1 Data Cleaning

Raw sensor values contain physically impossible readings: speed up to 516 km/h (world-record football sprint is ~36-44 km/h), HR up to 289 BPM (human max ~200-220), SpO2 as low as 54% (normal resting is 95-100%). We clip to realistic football-physiology ranges before plotting.

In [ ]:
df['Speed_clean']  = df['Speed (km/h)'].clip(0, 36)
df['HR_clean']     = df['Heart Rate (BPM)'].clip(60, 220)
df['SpO2_clean']   = df['SpO2 (%)'].clip(80, 100)
df['Accel_clean']  = df['Acceleration (m/s\u00b2)'].clip(0, 20)
print('Raw max speed:', df['Speed (km/h)'].max(), '-> cleaned max:', df['Speed_clean'].max())
print('Raw max HR:', df['Heart Rate (BPM)'].max(), '-> cleaned max:', df['HR_clean'].max())
print('Raw min SpO2:', df['SpO2 (%)'].min(), '-> cleaned min:', df['SpO2_clean'].min())
df[['Speed_clean','HR_clean','SpO2_clean','Accel_clean']].describe()


## 3 · Visualization Style

In [ ]:
DARK_BG  = '#0a0a1a'
PANEL_BG = '#111125'
TEXT_COL = '#e0e0e0'
GRID_COL = '#2a2a4a'
zone_order  = ['Walking','Jogging','Running','High-speed Running','Sprinting']
zone_colors = {
    'Walking':            '#00BFFF',
    'Jogging':            '#00CC44',
    'Running':            '#FFAA00',
    'High-speed Running': '#FF4400',
    'Sprinting':          '#FF0066'
}
def style_ax(ax, xlabel='', ylabel='', title=''):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=TEXT_COL, labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_COL)
    ax.grid(True, color=GRID_COL, linewidth=0.5, alpha=0.6)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_COL, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_COL, fontsize=9)
    if title:  ax.set_title(title, color=TEXT_COL, fontsize=10, pad=6)
print('Style helpers defined')


---
## Q1 — Speed Zone Distribution

**Chart:** Bar chart

**Encoding:**
- *y-axis* → count (proxy for time spent at 100 ms sampling rate)
- *x-axis* → Speed Zone, ordered low → high intensity
- *colour* → cool-to-warm gradient reinforcing intensity

**Why:** Bar height is a pre-attentive visual channel, making it trivial to compare proportions across categories at a glance. The hue gradient encodes the ordinal intensity ordering (blue = low effort → red = maximal effort).

**Interpretation:** The player spent the majority of session time in walking and jogging zones, with sprinting representing a small fraction. This is consistent with interval training patterns where brief high-intensity bursts alternate with extended recovery.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK_BG)
fig.suptitle('Speed Zone Distribution', color=TEXT_COL, fontsize=13)
zc = df['Speed Zone'].value_counts().reindex(zone_order).fillna(0)
zp = zc / zc.sum() * 100
bars = ax.bar(zc.index, zc.values, color=[zone_colors[z] for z in zc.index], edgecolor='none')
for b, c, p in zip(bars, zc.values, zp.values):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + c * 0.02,
            f'{p:.1f}%', ha='center', va='bottom', color=TEXT_COL, fontsize=10)
ax.set_ylabel('Count (sample frequency)', fontsize=10, color=TEXT_COL)
style_ax(ax)
plt.tight_layout()
plt.show()
print(f'Most common zone: {zc.idxmax()} ({zp.max():.1f}%)')


## Q2 — Heart Rate vs Exercise Intensity

**Chart:** Scatter plot with binned trend line

**Encoding:**
- *x-axis* → Speed (km/h)
- *y-axis* → Heart Rate (BPM)
- *point colour* → Speed Zone
- *white line* → binned mean trend

**Why:** Scatter plots map two continuous variables to x/y position, revealing correlation, clusters, and outliers. We subsample to 3% of data to avoid overplotting (52k+ points would otherwise render as a solid blob). The LOESS-like binned trend line shows the central trend.

**Interpretation:** Heart rate shows a moderate positive correlation with speed, but the trend is non-linear. HR rises sharply at low-to-moderate speeds then plateaus at higher speeds — a classic cardiovascular response pattern with inherent physiological lag.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), facecolor=DARK_BG)
fig.suptitle('Heart Rate vs Speed (Exercise Intensity)', color=TEXT_COL, fontsize=13)
s = df.sample(frac=0.03, random_state=42)
for z in zone_order:
    zd = s[s['Speed Zone'] == z]
    ax.scatter(zd['Speed_clean'], zd['HR_clean'], s=1, alpha=0.4,
               color=zone_colors[z], label=z, linewidths=0)
# Binned trend line
bins = pd.cut(df['Speed_clean'], bins=np.arange(0, 37, 2))
trend = df.groupby(bins)['HR_clean'].mean()
tx = trend.index.map(lambda x: x.mid)
ax.plot(tx, trend.values, 'w-', linewidth=2, label='Trend (binned mean)', zorder=10)
ax.set_xlabel('Speed (km/h)', fontsize=10, color=TEXT_COL)
ax.set_ylabel('Heart Rate (BPM)', fontsize=10, color=TEXT_COL)
ax.legend(fontsize=7, loc='upper left', facecolor='#1a1a2e',
          edgecolor=GRID_COL, labelcolor=TEXT_COL)
style_ax(ax)
plt.tight_layout()
plt.show()
r = df['Speed_clean'].corr(df['HR_clean'])
print(f'Pearson r(Speed, HR) = {r:.3f}')


## Q3 — Distance Covered & Fatigue Analysis

**Chart:** Dual line plot (two y-axes)

**Encoding:**
- *x-axis* → elapsed time (minutes)
- *y-axis (left, blue)* → cumulative total distance (m)
- *y-axis (right, orange)* → rolling mean speed (60-sample ≈ 6 s window)

**Why:** A line plot is the standard for temporal trends. The cumulative distance line reveals pace — flat sections indicate rest/walking, steep sections indicate running/sprinting. The rolling speed panel detects fatigue (progressive speed decrease across repeated efforts).

**Interpretation:** The step-like distance pattern confirms structured interval training — flat periods (recovery) alternate with steep climbs (active running). Rolling speed peaks help identify whether the player maintains output or slows over time (fatigue).

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(14, 7), facecolor=DARK_BG,
                            gridspec_kw={'height_ratios': [1, 1], 'hspace': 0.1})
fig.suptitle('Distance Covered & Fatigue Analysis', color=TEXT_COL, fontsize=13)
# Cumulative distance
a1.fill_between(df['Elapsed_min'], df['Total Distance (m)'], color='#00BFFF', alpha=0.3)
a1.plot(df['Elapsed_min'], df['Total Distance (m)'], color='#00BFFF', linewidth=1.0)
a1.set_ylabel('Cumulative Distance (m)', fontsize=10, color=TEXT_COL)
style_ax(a1)
a1.set_xticklabels([])
# Rolling mean speed (fatigue indicator)
rs = df['Speed_clean'].rolling(60, min_periods=1).mean()
a2.fill_between(df['Elapsed_min'], rs, color='#FFAA00', alpha=0.3)
a2.plot(df['Elapsed_min'], rs, color='#FFAA00', linewidth=1.0)
a2.set_xlabel('Elapsed Time (min)', fontsize=10, color=TEXT_COL)
a2.set_ylabel('Rolling Avg Speed (km/h)', fontsize=10, color=TEXT_COL)
style_ax(a2)
plt.tight_layout()
plt.show()
print(f'Total distance covered: {df["Total Distance (m)"].max():.1f} m')
print(f'Average speed: {df["Speed_clean"].mean():.1f} km/h')


## Q4 — Sprint Frequency & Duration

**Chart:** Combo bar + line

**Encoding:**
- *x-axis* → minute of session
- *y-axis (bars, pink)* → sprint sample count per minute (proxy for duration)
- *y-axis (line, yellow)* → max sprint speed reached that minute

**Why:** Bars show how many sprinting samples occurred each minute — more samples means longer sprint duration. The overplotted line adds peak intensity context per minute, revealing whether sprint quality was maintained or declined.

**Interpretation:** Sprint events cluster in distinct bursts (~4-5 intervals), with peak speeds varying across bursts. Consistent peak speeds suggest maintained sprint quality; declining peaks would indicate velocity-specific fatigue.

In [ ]:
df['is_sprint'] = (df['Speed Zone'] == 'Sprinting').astype(int)
df['minute'] = df['Elapsed_min'].astype(int)
sp = df.groupby('minute')['is_sprint'].sum()
ms = df[df['is_sprint'] == 1].groupby('minute')['Speed_clean'].max()
fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK_BG)
fig.suptitle('Sprint Frequency & Peak Speed per Minute', color=TEXT_COL, fontsize=13)
ax.bar(sp.index, sp.values, color='#FF0066', alpha=0.7, edgecolor='none',
       label='Sprint samples')
ax2 = ax.twinx()
ax2.plot(ms.index, ms.values, color='#FFDD44', linewidth=2, marker='o',
         markersize=5, label='Max sprint speed (km/h)')
ax.set_xlabel('Minute of Session', fontsize=10, color=TEXT_COL)
ax.set_ylabel('Sprint samples', fontsize=10, color=TEXT_COL)
ax2.set_ylabel('Max speed (km/h)', fontsize=10, color='#FFDD44')
l1, l2 = ax.get_legend_handles_labels()
l3, l4 = ax2.get_legend_handles_labels()
ax2.legend(l1 + l3, l2 + l4, fontsize=8, loc='upper left',
           facecolor='#1a1a2e', edgecolor=GRID_COL, labelcolor=TEXT_COL)
style_ax(ax)
plt.tight_layout()
plt.show()
print(f'Total sprint samples: {int(sp.sum())}')
if len(ms) > 0:
    print(f'Peak sprint speed: {ms.max():.1f} km/h')
    print(f'Mean peak speed: {ms.mean():.1f} km/h')


## Q5 — Acceleration / Deceleration Balance

**Chart:** Grouped bar chart

**Encoding:**
- *x-axis* → minute of session
- *y-axis* → cumulative event count
- *colour* → green = acceleration, red = deceleration

**Why:** Side-by-side bars make imbalance visually obvious. A large accel/decel imbalance indicates the training focus — more accels suggest straight-line sprint drills; more decels suggest change-of-direction work.

**Interpretation:** A roughly symmetric accel/decel profile (ratio near 1.0) indicates balanced start/stop actions, typical of multi-directional drills or small-sided games.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK_BG)
fig.suptitle('Acceleration vs Deceleration Balance', color=TEXT_COL, fontsize=13)
ca = df.groupby('minute')['Accel Events'].last().fillna(method='ffill')
cd_val = df.groupby('minute')['Decel Events'].last().fillna(method='ffill')
x = np.arange(len(ca))
w = 0.35
ax.bar(x - w / 2, ca.values, w, color='#00CC44', edgecolor='none', label='Accel Events')
ax.bar(x + w / 2, cd_val.values, w, color='#FF4466', edgecolor='none', label='Decel Events')
ax.set_xticks(x)
ax.set_xticklabels(ca.index.astype(str), fontsize=8)
ax.set_ylabel('Cumulative count', fontsize=10, color=TEXT_COL)
ax.legend(fontsize=9, loc='upper left', facecolor='#1a1a2e',
          edgecolor=GRID_COL, labelcolor=TEXT_COL)
style_ax(ax)
plt.tight_layout()
plt.show()
ta = int(df['Accel Events'].iloc[-1])
td = int(df['Decel Events'].iloc[-1])
print(f'Total Accel Events: {ta}')
print(f'Total Decel Events: {td}')
if td > 0:
    print(f'Accel / Decel Ratio: {ta / td:.2f}')


## Q6 — Player Load Accumulation

**Chart:** Filled line chart with linear trend

**Encoding:**
- *x-axis* → elapsed time (min)
- *y-axis* → cumulative PlayerLoad (arbitrary units, AU)
- *filled area* → visual emphasis on accumulation
- *dashed line* → linear regression trend

**Why:** A filled area chart makes the accumulating nature of PlayerLoad explicit — the growing area visually communicates 'load added over time'. The trend line slope indicates the mechanical stress rate; a steeper slope correlates with higher injury risk.

**Interpretation:** A roughly linear PlayerLoad accumulation indicates sustained, consistent mechanical stress throughout the session — expected for interval training with repeated high-intensity efforts.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5), facecolor=DARK_BG)
fig.suptitle('Cumulative PlayerLoad (Mechanical Stress Accumulation)',
             color=TEXT_COL, fontsize=13)
ax.fill_between(df['Elapsed_min'], df['PlayerLoad (cumulative)'],
                color='#8A2BE2', alpha=0.25)
ax.plot(df['Elapsed_min'], df['PlayerLoad (cumulative)'],
        color='#8A2BE2', linewidth=1.5, label='Cumulative PlayerLoad')
# Linear trend
z = np.polyfit(df['Elapsed_min'], df['PlayerLoad (cumulative)'], 1)
p = np.poly1d(z)
ax.plot(df['Elapsed_min'], p(df['Elapsed_min']), 'r--', linewidth=1.2,
        alpha=0.8, label=f'Trend (slope: {z[0]:.2f} AU/min)')
ax.set_xlabel('Elapsed Time (min)', fontsize=10, color=TEXT_COL)
ax.set_ylabel('PlayerLoad (cumulative, AU)', fontsize=10, color=TEXT_COL)
ax.legend(fontsize=9, loc='upper left', facecolor='#1a1a2e',
          edgecolor=GRID_COL, labelcolor=TEXT_COL)
style_ax(ax)
plt.tight_layout()
plt.show()
print(f'Final PlayerLoad: {df["PlayerLoad (cumulative)"].iloc[-1]:.1f} AU')
print(f'Average accumulation rate: {z[0]:.2f} AU/min')


## Q7 — GPS-Based Movement Analysis

**Chart:** Pitch map / heatmap (if GPS available)

**Encoding:**
- *x-axis* → longitude (converted to metres)
- *y-axis* → latitude (converted to metres)
- *colour* → 2D density (heatmap)

**Why:** Spatial encoding mirrors the physical pitch, making it intuitive to identify where the player spends time. A heatmap shows density of presence by zone.

**Data limitation:** All GPS coordinates in this dataset are (0.0, 0.0). The GPS sensor was either not calibrated, not connected, or positional data was not recorded during this session. Without GPS, tactical analysis (zone coverage, distance from teammates, heatmaps) is impossible. This is a critical data quality issue that should be flagged at ingestion.

In [ ]:
vg = df[(df['Latitude'] != 0) | (df['Longitude'] != 0)]
print(f'Valid GPS rows: {len(vg)} out of {len(df)} ({len(vg) / len(df) * 100:.1f}%)')
print(f'Latitude  range: [{df["Latitude"].min():.6f}, {df["Latitude"].max():.6f}]')
print(f'Longitude range: [{df["Longitude"].min():.6f}, {df["Longitude"].max():.6f}]')
fig, ax = plt.subplots(figsize=(8, 5), facecolor=DARK_BG)
if vg.empty:
    ax.text(0.5, 0.5,
            'No valid GPS data available.\n'
            'All coordinates are (0.0, 0.0).\n\n'
            'The GPS sensor was not recording positional data\n'
            'during this session.\n'
            'A movement map cannot be generated.',
            ha='center', va='center', fontsize=12, color=TEXT_COL,
            transform=ax.transAxes)
    ax.set_title('GPS Data Unavailable', color=TEXT_COL, fontsize=13)
    ax.axis('off')
else:
    ax.text(0.5, 0.5, 'GPS data found - heatmap would be plotted here.',
            ha='center', va='center', fontsize=12, color=TEXT_COL,
            transform=ax.transAxes)
    ax.set_title('GPS Movement Map', color=TEXT_COL, fontsize=13)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Q8 — Safety Alerts: High Heart Rate + Low SpO₂

**Chart:** Scatter plot with coloured risk quadrants

**Encoding:**
- *x-axis* → SpO₂ (%)
- *y-axis* → Heart Rate (BPM)
- *background* → red (danger), yellow (caution), green (safe) quadrants
- *point colour* → severity classification per point

**Why:** Quadrants defined by clinical safety thresholds (HR > 180, SpO₂ < 85) make danger points instantly visible. Semantic colours (green/yellow/red) leverage universal risk associations, reducing cognitive load for coaches interpreting the plot in real time.

**Interpretation:** Points in the top-left quadrant (HR > 180 AND SpO₂ < 85) are critical danger. Any point with HR > 200 or SpO₂ < 85 independently warrants a coach alert. The summary counts show what fraction of the session was in each risk category.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7), facecolor=DARK_BG)
fig.suptitle('Safety Alert: Heart Rate vs SpO\u2082', color=TEXT_COL, fontsize=13)
s = df.sample(frac=0.05, random_state=99)
# Risk quadrant backgrounds
ax.axvspan(80, 85, 180, 220, color='#FF0000', alpha=0.12)  # danger zone
ax.axvspan(80, 90, 0, 220, color='#FFAA00', alpha=0.05)    # SpO2 caution
ax.axhspan(180, 220, 80, 100, color='#FFAA00', alpha=0.05)  # HR caution
# Classify each point
safe = s[(s['SpO2_clean'] >= 90) & (s['HR_clean'] < 180)]
caut = s[((s['SpO2_clean'] >= 85) & (s['SpO2_clean'] < 90)) |
         ((s['HR_clean'] >= 180) & (s['HR_clean'] < 200))]
dang = s[(s['SpO2_clean'] < 85) | (s['HR_clean'] >= 200)]
ax.scatter(safe['SpO2_clean'], safe['HR_clean'], s=2, alpha=0.5,
           color='#00CC44', label=f'Safe ({len(safe)})', linewidths=0)
ax.scatter(caut['SpO2_clean'], caut['HR_clean'], s=2, alpha=0.5,
           color='#FFAA00', label=f'Caution ({len(caut)})', linewidths=0)
ax.scatter(dang['SpO2_clean'], dang['HR_clean'], s=4, alpha=0.7,
           color='#FF0044', label=f'Danger ({len(dang)})', linewidths=0)
# Threshold lines
ax.axvline(85, color='red', ls='--', alpha=0.6)
ax.axvline(90, color='orange', ls='--', alpha=0.4)
ax.axhline(180, color='orange', ls='--', alpha=0.4)
ax.axhline(200, color='red', ls='--', alpha=0.4)
ax.set_xlabel('SpO\u2082 (%)', fontsize=10, color=TEXT_COL)
ax.set_ylabel('Heart Rate (BPM)', fontsize=10, color=TEXT_COL)
ax.legend(fontsize=8, loc='lower right', facecolor='#1a1a2e',
          edgecolor=GRID_COL, labelcolor=TEXT_COL)
style_ax(ax)
plt.tight_layout()
plt.show()
# Full-dataset safety counts
fs = len(df[(df['SpO2_clean'] >= 90) & (df['HR_clean'] < 180)])
fc = len(df[((df['SpO2_clean'] >= 85) & (df['SpO2_clean'] < 90)) |
             ((df['HR_clean'] >= 180) & (df['HR_clean'] < 200))])
fd = len(df[(df['SpO2_clean'] < 85) | (df['HR_clean'] >= 200)])
total = len(df)
print('=== SAFETY ALERT SUMMARY (full dataset) ===')
print(f'Safe:    {fs} ({fs / total * 100:.1f}%)')
print(f'Caution: {fc} ({fc / total * 100:.1f}%)')
print(f'Danger:  {fd} ({fd / total * 100:.1f}%)')


## Visual Encoding & Design Summary

| Q | Chart Type | Position Channel | Colour Channel | Design Rationale |
|---|---|---|---|---|
| 1 | Bar chart | y = count | hue = intensity gradient | Categorical comparison; bar height is pre-attentive |
| 2 | Scatter | x = speed, y = HR | hue = zone | Reveals correlation; opacity avoids overplotting |
| 3 | Line (dual axis) | x = time | hue = metric | Two temporal trends on shared x-axis |
| 4 | Combo bar + line | x = minute | hue = bar | Frequency + peak value in one view |
| 5 | Grouped bar | x = minute, y = count | hue = accel/decel | Side-by-side balance comparison |
| 6 | Filled line | x = time, y = load | single hue | Emphasises accumulation over time |
| 7 | Heatmap (N/A) | N/A | N/A | GPS data unavailable — documented limitation |
| 8 | Scatter + quadrants | x = SpO₂, y = HR | hue = risk level | Quadrants map to clinical safety thresholds |

**Colour palette rationale:** Speed zones follow a perceptually uniform cool-to-warm gradient (blue → green → yellow → orange → red) matching effort intensity. The safety plot uses semantic colours (green/yellow/red) that map to universal risk associations.

**Dark theme:** All plots share a dark background to maximise contrast for thin time-series lines and make saturated colour encodings pop. Grid lines use low-opacity lines to aid reading without cluttering.

---

## Key Findings

1. **Speed zones** dominated by walking/recovery — consistent with interval training.
2. **Heart rate** shows moderate positive correlation with speed but with physiological lag.
3. **Distance accumulation** is step-like, confirming structured sprint/rest intervals.
4. **Sprint events** cluster in distinct bursts (≈4–5 intervals).
5. **Accel/decel ratio** is roughly symmetric, suggesting multi-directional drill work.
6. **PlayerLoad** accumulates approximately linearly — sustained mechanical stress.
7. **GPS data** was unavailable (all zeros) — a data quality gap preventing tactical analysis.
8. **Safety analysis** flags danger-zone samples that should trigger real-time coach alerts.

### Data Quality Notes
- Raw speed and HR values contained physically impossible readings; all analyses used clipped values.
- GPS sensor produced no positional data.
- A production system should include real-time validation rules (speed < 40 km/h, HR < 220 BPM, SpO₂ > 80%) to catch sensor errors at ingestion.

### Deliverables Met
- All 8 analysis questions answered with code + visualisation
- Visual encodings (position, colour, size) documented for each plot
- Design rationale explained (dark theme, gradient palettes, chart type selection)
- Data quality issues identified and cleaning strategy described
